# 17 · Model Development & Evaluation — Climb, Track, Slice

**Part 2 · Machine Learning Systems** · *Prerequisites: 15–16* · *Book mapping: DMLS ch. 6*

Model development in production is not a search for the cleverest architecture; it is a
disciplined climb from trivial baselines, with every step **tracked** (so results are
reproducible and comparable) and **evaluated past the average** (so the failures hiding
inside a good aggregate metric get found before users find them).

**Learning objectives.**

1. Build a minimal **experiment tracker** and run everything through it.
2. Climb the **baseline ladder** — majority → heuristic → linear → boosted trees — and
   see where the gains actually come from.
3. Evaluate like an adult: **calibration** (and fixing it), **slice-based evaluation**
   (finding the segment your average hides), **behavioral tests** (invariance and
   monotonicity), and **bootstrap uncertainty** on model comparisons.
4. Keep experiments reproducible: seeds, data versions, and the one-line habits that make
   "which model is in prod?" answerable.

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier

from sysdes import scratch_dir, viz

viz.use_style()
rng = np.random.default_rng(seed=17)
RUNS = scratch_dir("17-experiments")

## 1. The dataset (with a planted weakness)

Synthetic credit-risk data with a real interaction structure — and, deliberately, a small
**mobile-new-user segment where the pattern differs**, so slice evaluation later has
something true to find:

In [ ]:
N = 14_000
df = pd.DataFrame({
    "amount": rng.exponential(400, N).round(2),
    "income": rng.lognormal(10.4, 0.5, N).round(2),
    "tenure_months": rng.gamma(3, 8, N).round(1),
    "is_mobile": (rng.random(N) < 0.35).astype(int),
    "age": rng.normal(40, 12, N).clip(18, 80).round(0),
})
new_mobile = (df.is_mobile == 1) & (df.tenure_months < 6)      # ~5% of users

risk = (0.0004 * df.amount - 0.15 * np.log(df.income / 20_000)
        - 0.008 * df.tenure_months
        # The dominant signal is an AND-shaped interaction - the kind linear
        # models cannot represent (nb 16's crossing lesson, from the data's side):
        + 1.7 * ((df.amount > 400) & (df.tenure_months < 18))
        + rng.normal(0, 0.7, N))
# In the new-mobile segment the drivers differ (fraud rings exploit new devices):
risk[new_mobile] = 1.1 * rng.normal(0, 1, int(new_mobile.sum())) + 0.4
df["default"] = (risk > 0.9).astype(int)

FEATURES = ["amount", "income", "tenure_months", "is_mobile", "age"]
train, test = df.iloc[:10_000], df.iloc[10_000:]
print(f"default rate: {df.default.mean():.1%}; "
      f"new-mobile segment: {new_mobile.mean():.1%} of rows")

## 2. Build: an experiment tracker

Before any model: the bookkeeping. Every serious question you'll ask later — *which run
is in prod? what data was it trained on? did the new feature help?* — is unanswerable
without a run ledger. Real tools (MLflow, W&B) are this class with a UI:

In [ ]:
class ExperimentTracker:
    """Append-only ledger of runs: params + metrics + provenance. (A log as the
    source of truth and views derived from it - nb 12 knew this shape.)"""

    def __init__(self, path: Path):
        self.path = path / "runs.jsonl"

    def log(self, name: str, params: dict, metrics: dict, **provenance) -> str:
        run_id = f"run-{int(time.time()*1e6)}"
        record = {"run_id": run_id, "name": name, "params": params,
                  "metrics": metrics, **provenance}
        with open(self.path, "a") as f:
            f.write(json.dumps(record) + "\n")
        return run_id

    def runs(self) -> pd.DataFrame:
        rows = [json.loads(line) for line in open(self.path)] if self.path.exists() else []
        return pd.json_normalize(rows)

    def best(self, metric: str) -> pd.Series:
        runs = self.runs()
        return runs.loc[runs[f"metrics.{metric}"].idxmax()]

tracker = ExperimentTracker(RUNS)
DATA_VERSION = "credit-synth-v1@seed17"     # in real life: a lake snapshot/commit hash

def evaluate_and_log(name, params, predict_proba_fn):
    p = predict_proba_fn(test[FEATURES])
    auc = roc_auc_score(test.default, p)
    tracker.log(name, params, {"auc": round(float(auc), 4)},
                data_version=DATA_VERSION, n_train=len(train))
    return auc

What to log, always: **params** (all of them), **metrics**, **data version**, **code
version** (a git hash — our stand-in string plays the role), and the **seed**. The test
of your logging: could a teammate reproduce the run from the ledger row alone?

## 3. The baseline ladder

Every rung must beat the one below *on the tracked metric* to justify its added
complexity. Rung 1 costs nothing and calibrates expectations; rung 2 is what the business
would do without you; the gap each rung adds is *your measured value*:

In [ ]:
ladder = {}

# Rung 1 - chance: predict the base rate for everyone.
ladder["1-baserate"] = evaluate_and_log(
    "baserate", {}, lambda X: np.full(len(X), train.default.mean()))

# Rung 2 - the analyst's rule: big amount relative to income = risky.
ladder["2-heuristic"] = evaluate_and_log(
    "heuristic", {"rule": "amount/income"},
    lambda X: (X.amount / X.income).to_numpy())

# Rung 3 - logistic regression on standardized features.
mu, sd = train[FEATURES].mean(), train[FEATURES].std()
lr = LogisticRegression(max_iter=1000).fit((train[FEATURES] - mu) / sd, train.default)
ladder["3-logistic"] = evaluate_and_log(
    "logistic", {"model": "LR"}, lambda X: lr.predict_proba((X - mu) / sd)[:, 1])

# Rung 4 - gradient-boosted trees (the tabular workhorse).
gbm = HistGradientBoostingClassifier(random_state=0).fit(train[FEATURES], train.default)
ladder["4-gbm"] = evaluate_and_log(
    "gbm", {"model": "HistGBM", "defaults": True},
    lambda X: gbm.predict_proba(X)[:, 1])

for rung, auc in ladder.items():
    print(f"{rung:12s} AUC {auc:.3f}")
assert ladder["1-baserate"] < 0.55, "chance is chance"
assert ladder["2-heuristic"] > 0.6, "one good ratio already carries signal"
assert ladder["3-logistic"] > ladder["2-heuristic"]
assert ladder["4-gbm"] > ladder["3-logistic"], "trees find the interaction LR can't"

gain_lr = ladder["3-logistic"] - ladder["2-heuristic"]
gain_gbm = ladder["4-gbm"] - ladder["3-logistic"]
print(f"\ngain from learning weights (LR): +{gain_lr:.3f}; "
      f"from learning interactions (GBM): +{gain_gbm:.3f}")

The shape of that ladder is the shape of most tabular ML: the heuristic captures the
headline signal, the linear model tidies it, and the tree ensemble collects the
interactions — a meaningful but *bounded* increment. Which is why chasing exotic
architectures before exhausting features (nb 16) and data (nb 15) is usually backwards —
and why "we can't beat the heuristic" is a legitimate, money-saving finding. (Ensembles
push further by combining *diverse* errors — bagging averages variance away, boosting
chases residuals, stacking learns the combination; our GBM is boosting already.)

And the cheapest experiment of all — reproducibility:

In [ ]:
gbm_again = HistGradientBoostingClassifier(random_state=0).fit(train[FEATURES], train.default)
assert np.allclose(gbm.predict_proba(test[FEATURES]), gbm_again.predict_proba(test[FEATURES])), \
    "same code + same data + same seed = same model, bit for bit"
print("reproducibility check passed - if this ever fails, no other number can be trusted")

## 4. Evaluation past the average

### 4.1 Calibration — do the probabilities mean anything?

Notebook 13 turned probabilities into money with thresholds. That math assumed `p=0.3`
means "30% of such cases default" — **calibration**. Rankings (AUC) can be perfect while
probabilities are fantasy. The reliability diagram groups predictions into bins and
compares claimed vs actual rates; **ECE** (expected calibration error) is its weighted
average gap:

In [ ]:
def reliability(y, p, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    claimed = np.array([p[idx == b].mean() if (idx == b).any() else np.nan
                        for b in range(bins)])
    actual = np.array([y[idx == b].mean() if (idx == b).any() else np.nan
                       for b in range(bins)])
    weights = np.array([(idx == b).mean() for b in range(bins)])
    mask = ~np.isnan(claimed)
    ece = float(np.nansum(weights[mask] * np.abs(claimed - actual)[mask]))
    return claimed, actual, ece

# An overfit deep tree: great at memorizing, awful at probability.
tree = DecisionTreeClassifier(random_state=0).fit(train[FEATURES], train.default)
p_tree = tree.predict_proba(test[FEATURES])[:, 1]
# The same tree, wrapped in isotonic calibration (fit on held-out folds):
calibrated = CalibratedClassifierCV(
    DecisionTreeClassifier(random_state=0), method="isotonic", cv=3
).fit(train[FEATURES], train.default)
p_cal = calibrated.predict_proba(test[FEATURES])[:, 1]

y = test.default.to_numpy()
claimed_t, actual_t, ece_tree = reliability(y, p_tree)
claimed_c, actual_c, ece_cal = reliability(y, p_cal)
print(f"ECE - raw overfit tree: {ece_tree:.3f}, isotonic-calibrated: {ece_cal:.3f}")
assert ece_cal < ece_tree / 2, "calibration fixed the probabilities"

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot([0, 1], [0, 1], ":", color="#9e9e9e", label="perfect")
ax.plot(claimed_t, actual_t, "o-", label=f"overfit tree (ECE {ece_tree:.2f})")
ax.plot(claimed_c, actual_c, "s-", label=f"calibrated (ECE {ece_cal:.2f})")
ax.set_xlabel("claimed probability")
ax.set_ylabel("actual default rate")
ax.set_title("Reliability diagram: can the probabilities be taken literally?")
ax.legend()
plt.show()

Calibration is monotone — it never changes the *ranking* (AUC untouched) — but it changes
every **decision** computed from probabilities: thresholds, expected costs, bids. If
nb 13's profit math runs on uncalibrated scores, the "optimal" threshold is optimal for a
fictional world.

### 4.2 Slice-based evaluation — what the average hides

One number over 4,000 test rows is a democracy where big segments outvote small ones.
Slice the metric by the segments that matter (platforms, tenure cohorts, geographies,
protected groups) and the planted weakness surfaces:

In [ ]:
test_slices = {
    "ALL": np.ones(len(test), bool),
    "web": (test.is_mobile == 0).to_numpy(),
    "mobile, tenure>=6": ((test.is_mobile == 1) & (test.tenure_months >= 6)).to_numpy(),
    "mobile, tenure<6": ((test.is_mobile == 1) & (test.tenure_months < 6)).to_numpy(),
}
p_gbm = gbm.predict_proba(test[FEATURES])[:, 1]
report = pd.DataFrame([
    {"slice": name, "n": int(m.sum()),
     "auc": round(roc_auc_score(y[m], p_gbm[m]), 3),
     "default_rate": round(float(y[m].mean()), 3)}
    for name, m in test_slices.items() if len(np.unique(y[m])) == 2
])
print(report.to_string(index=False))

auc_all = report.loc[report["slice"] == "ALL", "auc"].item()
auc_weak = report.loc[report["slice"] == "mobile, tenure<6", "auc"].item()
assert auc_weak < auc_all - 0.1, \
    "a segment where the model is near-blind hid inside a healthy average"

The overall AUC said "ship it"; the slice report says "not to new mobile users, where the
drivers are different and the data thin". Responses, in order of honesty: get segment
data/features (nb 15's active learning aimed there), a segment-specific model or
interaction features, a higher decision threshold for the weak slice (route to manual
review — nb 13's policy layer), or at minimum monitoring that slice separately in
production (nb 19). When slices are protected groups, this table *is* the first fairness
audit — same mechanics, higher stakes.

### 4.3 Behavioral tests — unit tests for models

Metrics measure aggregate skill; **behavioral tests** check properties every stakeholder
assumes: predictions shouldn't change when nothing relevant changes (**invariance**), and
should move the right *direction* when one thing changes (**monotonicity**). Trees happily
violate the second in noisy regions — unless constrained:

In [ ]:
def directional_violations(model) -> int:
    """Sweep amount over a grid for 200 fixed customers; risk must not DROP
    as the requested amount grows, all else equal."""
    base = test[FEATURES].iloc[:200].copy()
    grid = np.linspace(50, 3_000, 12)
    violations = 0
    for _, row in base.iterrows():
        probe = pd.DataFrame([row] * len(grid))
        probe["amount"] = grid
        p = model.predict_proba(probe[FEATURES])[:, 1]
        violations += int((np.diff(p) < -1e-9).any())
    return violations

mono = HistGradientBoostingClassifier(
    random_state=0,
    monotonic_cst=[1, 0, 0, 0, 0],       # amount: risk may only rise with it
).fit(train[FEATURES], train.default)

v_free = directional_violations(gbm)
v_mono = directional_violations(mono)
auc_mono = roc_auc_score(y, mono.predict_proba(test[FEATURES])[:, 1])
print(f"customers whose risk DROPS somewhere as amount rises - "
      f"unconstrained: {v_free}/200, monotonic: {v_mono}/200 "
      f"(AUC {ladder['4-gbm']:.3f} -> {auc_mono:.3f})")
assert v_mono == 0, "the constraint makes the sane property hold by construction"

# Invariance: age is (by construction here) irrelevant to risk - so a 1-year
# nudge should not move anyone's score. Test it:
probe = test[FEATURES].iloc[:500].copy()
p_before = mono.predict_proba(probe)[:, 1]
drift = np.abs(mono.predict_proba(probe.assign(age=probe.age + 1))[:, 1] - p_before).max()
print(f"max prediction change from a 1-year age nudge: {drift:.3f}")
assert drift > 0.05, \
    "the invariance test CAUGHT the model leaning on a feature we believe irrelevant"

# The model fails the test: it found spurious splits on age (trees will use any
# noise you feed them). The strongest fix for a should-be-ignored feature is
# structural - don't feed it:
LEAN = [f for f in FEATURES if f != "age"]
lean_model = HistGradientBoostingClassifier(
    random_state=0, monotonic_cst=[1, 0, 0, 0]).fit(train[LEAN], train.default)
auc_lean = roc_auc_score(y, lean_model.predict_proba(test[LEAN])[:, 1])
print(f"without age: AUC {auc_mono:.3f} -> {auc_lean:.3f}; "
      f"age-invariance now holds by construction")
assert abs(auc_lean - auc_mono) < 0.01, \
    "the 'feature' contributed nothing but behavioral risk - dropping it was free"

The unconstrained model told some customers "ask for *more* money and your risk score
drops"; the tuned one quietly moved scores with birthdays. Both are lawsuit generators no
AUC would ever reveal — and both were caught by five-line tests. Behavioral tests belong
in CI next to your unit tests: every candidate model passes them *before* metrics are
even compared. (When a should-be-irrelevant feature *does* carry metric value, you've
found either leakage (nb 16) or a fairness conversation — never just shrug.)

### 4.4 Is the improvement real? Bootstrap the difference

GBM beat LR by a few points of AUC — on *one* test sample. Resample the test set with
replacement and the difference gets an uncertainty interval:

In [ ]:
p_lr = lr.predict_proba((test[FEATURES] - mu) / sd)[:, 1]
diffs = []
idx = np.arange(len(test))
for _ in range(400):
    b = rng.choice(idx, size=len(idx), replace=True)
    if len(np.unique(y[b])) < 2:
        continue
    diffs.append(roc_auc_score(y[b], p_gbm[b]) - roc_auc_score(y[b], p_lr[b]))
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"AUC(GBM) - AUC(LR): mean {np.mean(diffs):+.3f}, 95% CI [{lo:+.3f}, {hi:+.3f}]")
assert lo > 0, "the interval clears zero: this improvement survives resampling noise"

When the interval straddles zero, the honest conclusion is "not established — ship the
simpler model" (Occam with error bars). And note what this interval does **not** measure:
whether the better AUC makes more money — that requires the online experiments of nb 20.

## 5. Tuning, tracked (and kept in perspective)

Hyperparameter search is an outer loop around everything above — run it through the
tracker like any experiment. AutoML tools industrialize exactly this loop:

In [ ]:
val_train, val_holdout = train.iloc[:8_000], train.iloc[8_000:]
for lr_rate in (0.05, 0.1, 0.3):
    for depth in (3, None):
        m = HistGradientBoostingClassifier(
            learning_rate=lr_rate, max_depth=depth, random_state=0
        ).fit(val_train[FEATURES], val_train.default)
        auc_val = roc_auc_score(val_holdout.default,
                                m.predict_proba(val_holdout[FEATURES])[:, 1])
        tracker.log("gbm-tune", {"learning_rate": lr_rate, "max_depth": depth},
                    {"auc_val": round(float(auc_val), 4)}, data_version=DATA_VERSION)

runs = tracker.runs()
tune = runs[runs.name == "gbm-tune"]
best = tracker.best("auc_val")
print(tune[["params.learning_rate", "params.max_depth", "metrics.auc_val"]]
      .to_string(index=False))
print(f"\nbest: lr={best['params.learning_rate']}, depth={best['params.max_depth']}")
assert len(tune) == 6 and tune["metrics.auc_val"].max() >= tune["metrics.auc_val"].min()
spread = tune["metrics.auc_val"].max() - tune["metrics.auc_val"].min()
print(f"tuning spread: {spread:.3f} AUC - compare with the ladder's rungs above")

Note the proportions: the whole tuning grid moved the metric less than one rung of the
ladder. Tuning is real money at the margin and a distraction before the fundamentals —
data (nb 15), features (nb 16), framing (nb 13) — are exhausted. (On distributed
training: data parallelism — shard the batch, average gradients — and model parallelism —
shard the network — matter enormously for deep learning at scale; a tabular GBM on 14k
rows trains on a laptop in a second, and most business ML lives closer to this end than
the headline end.)

## 6. Exercises

1. **Ladder design.** Sketch the baseline ladder for "flag abusive chat messages":
   what is rung 1, what is the rung-2 heuristic the moderators already use, and what
   *specifically* must rung 3 beat to justify a model at all (metric AND operational
   cost)?
2. **Calibration under drift.** Your model is perfectly calibrated at launch. Nb 19-style
   covariate shift doubles the share of risky users. Is the *ranking* affected? Is the
   *calibration*? What does that do to nb 13's profit-optimal threshold?
3. **Slice discovery.** §4.2 checked slices we suspected. Describe two ways to find
   *unknown* weak slices from (features, prediction, error) triples — one manual (error
   analysis), one automatic (train-a-model-on-errors) — and the trap of testing dozens of
   slices at once.
4. **Leaderboard erosion.** A team evaluates 200 candidate models against the same test
   set over six months, always shipping the best. Why does the reported test AUC drift
   upward faster than true skill? Connect to §4.4 and name the fix.
5. **The prod ledger.** Design the minimal record your tracker must store so that, one
   year later, you can answer: "the March 3rd incident — which model was serving, trained
   on what data, and can we rebuild it bit-for-bit?" (nb 21 turns your answer into a
   model registry.)

### Solutions

**1.** Rung 1: flag nothing (or flag everything — whichever matches the current default);
rung 2: the moderators' keyword/regex list — it has known precision and, crucially,
**explainable decisions**; rung 3 must beat the list at equal *reviewer workload*
(precision at the same review volume, nb 13's budget framing) by enough to cover the new
operational costs: serving infra, monitoring, appeals for opaque decisions.

**2.** Ranking (AUC): unchanged by a shift in the *mix* if within-group behavior holds —
AUC is composition-robust... until concept shift. Calibration: **broken immediately** —
predicted 0.2 now corresponds to a different actual rate because prevalence moved
(calibration is prevalence-sensitive). The nb 13 threshold, computed from costs and
calibrated p, now sits in the wrong place: recalibrate (cheap, monotone) before touching
the model — often the whole "model degraded!" incident is *only* this.

**3.** Manual: read the top-loss examples, grouped by anything groupable — the oldest
and highest-yield ritual in ML. Automatic: fit a shallow tree to predict
`|error|` from features; its top splits *are* candidate weak slices (slice-finding à la
SliceFinder). Trap: 40 slices × noise = several "weak" slices by luck — §4.4's bootstrap
(or multiple-testing corrections) before believing any of them.

**4.** Each evaluation is a draw from the test-noise distribution; always shipping the
max of n draws selects upward noise — the test set is slowly being *fit* by the
selection process (nb 16's leakage, in slow motion, through human choices). Fix: a
final untouched holdout used once per quarter/launch, and CIs (§4.4) so "best" must
clear noise, not just top the list.

**5.** Per run: run_id; git commit of code; **data version** (immutable snapshot id of
the lake partition set, nb 14); full param dict + seed; environment (library versions);
metric report *including slices*; artifact hash of the serialized model; and — the piece
everyone forgets — the **feature definitions version** (nb 16's skew lesson: the model
is only meaningful with its exact feature pipeline). Registry = this ledger + lifecycle
states (nb 21).

## 7. Takeaways

- Climb a **ladder**: chance → heuristic → linear → trees. Each rung's gain is your
  measured value; most tabular value arrives by the third rung. Track every run with
  params + data version + seed, or your results are anecdotes.
- Evaluate past the average: **calibration** makes probabilities mean something (and is
  cheap to fix); **slices** find the users your average outvotes; **behavioral tests**
  (invariance, monotonic constraints) catch lawsuit-shaped bugs no metric sees;
  **bootstrap CIs** separate improvement from noise.
- Reproducibility (same seed, same bits) is the substrate under every claim.
- Tuning is the garnish, not the meal — its spread was smaller than one ladder rung.

**Further reading.** DMLS ch. 6; Ribeiro et al., *Beyond Accuracy: Behavioral Testing of
NLP Models with CheckList* (ACL 2020); Guo et al., *On Calibration of Modern Neural
Networks* (ICML 2017).

**Next:** [18 · Deployment & serving](18-deployment-serving.ipynb) — the model is good
and its goodness is documented. Now it must answer requests, fast, without falling over.